# Predictive Anayltics: Support Vector Machines

Approach for SVM:
– Simply start without a kernel. Then, gradually make your model complex by integrating different
kind of kernels. Also, use grid search to find optimal values for your hyperparameters.
– How good is your model? Evaluate your model’s performance and comment on its shortfalls.
– Show how you model’s performance varies as you increase or decrease temporal or spatial
resolution How does your performance change when you only use census tract as spatial units?
– How could the model be improved further? Explain some of the improvement levers that you might
focus on in a follow-up project.

In [37]:
DO_GRID_SEARCH = True
GRID_SAMPLE = 5000
SPATIAL_UNIT = "community" # options: census, hexa, community

In [38]:
import pandas as pd
import polars as pl
import numpy as np
import matplotlib as plt
import datetime
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV
from joblib import load, dump

## Preparations

In [39]:
# "Settings" / Decisions for the training data
if SPATIAL_UNIT == "census":    
    DATA_PATH_TRAIN = "../data/train_test_data/svm_census_train.parquet"
    DATA_PATH_VAL = "../data/train_test_data/svm_census_val.parquet"
    DATA_PATH_TEST = "../data/train_test_data/svm_census_test.parquet" 
elif SPATIAL_UNIT == "hexa":
    DATA_PATH_TRAIN = "../data/train_test_data/svm_hexa_train.parquet"
    DATA_PATH_VAL = "../data/train_test_data/svm_hexa_val.parquet"
    DATA_PATH_TEST = "../data/train_test_data/svm_hexa_test.parquet" 
elif SPATIAL_UNIT == "community": 
    DATA_PATH_TRAIN = "../data/train_test_data/svm_community_train.parquet"
    DATA_PATH_VAL = "../data/train_test_data/svm_community_val.parquet"
    DATA_PATH_TEST = "../data/train_test_data/svm_community_test.parquet" 
else:
    print("Warning: No type of Spatial Unit given, Used census tract")
    DATA_PATH_TRAIN = "../data/train_test_data/svm_census_train.parquet"
    DATA_PATH_VAL = "../data/train_test_data/svm_census_val.parquet"
    DATA_PATH_TEST = "../data/train_test_data/svm_census_test.parquet" 



MODEL_PATH = "../models"

# Target and feature selection
TARGET_COL = "trip_count"
EXCLUDE_COLS = [
    TARGET_COL,
    "datetime_hour",      # real timestamp would lead to much leakage
    "_split_bucket",      # only used for splitting
    "month", # cyclic feature is used instead
    "weekday", # cyclic feature is used instead
    "hour", # cyclic feature is used instead
    # all other taxi data columns must be excluded to prevent leakage
    "trip_seconds_sum",
    "trip_seconds_mean",
    "trip_seconds_min",
    "trip_seconds_max",
    "trip_miles_sum",
    "trip_miles_mean",
    "trip_miles_min",
    "trip_miles_max",
    "fare_sum",
    "fare_mean",
    "fare_min",
    "fare_max",
    "tips_sum",
    "tips_mean",
    "tips_min",
    "tips_max",
    "tolls_sum",
    "tolls_mean",
    "tolls_min",
    "tolls_max",
    "extras_sum",
    "extras_mean",
    "extras_min",
    "extras_max",
    "trip_total_sum",
    "trip_total_mean",
    "trip_total_min",
    "trip_total_max",
    "most_common_payment_type",
    "trip_demand",
    "date",
]

Load data and select features and target

In [40]:
# Load data
train = pl.scan_parquet(DATA_PATH_TRAIN)
val = pl.scan_parquet(DATA_PATH_VAL)
test = pl.scan_parquet(DATA_PATH_TEST)

In [41]:
train_df = train.collect()
val_df = val.collect()
test_df = test.collect()

train_df = train_df.to_pandas()
val_df = val_df.to_pandas()
test_df = test_df.to_pandas()

In [42]:
train_df.head()

,datetime_hour,month,weekday,hour,month_sin,month_cos,weekday_sin,weekday_cos,hour_sin,hour_cos,...,tolls_max,extras_sum,extras_mean,extras_min,extras_max,trip_total_sum,trip_total_mean,trip_total_min,trip_total_max,most_common_payment_type
0,2024-07-07 05:00:00,7,7,5,1.224647e-16,-1.0,-0.781831,0.623490,0.965926,0.258819,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
1,2024-07-10 02:00:00,7,3,2,1.224647e-16,-1.0,0.974928,-0.222521,0.500000,0.866025,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
2,2024-07-13 01:00:00,7,6,1,1.224647e-16,-1.0,-0.974928,-0.222521,0.258819,0.965926,...,0.0,5.0,5.0,5.0,5.0,65.0,65.0,65.0,65.0,Cash
3,2024-07-10 17:00:00,7,3,17,1.224647e-16,-1.0,0.974928,-0.222521,-0.965926,-0.258819,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
4,2024-07-10 15:00:00,7,3,15,1.224647e-16,-1.0,0.974928,-0.222521,-0.707107,-0.707107,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips


In [43]:
train_df = train_df.sample(n=20000, random_state=42)
train_df_grid = train_df.sample(n=GRID_SAMPLE, random_state=42)

In [44]:
# Create X and y
feature_cols = [
    col for col in train_df.columns
    if col not in EXCLUDE_COLS
]

# Community_area is a categorical id, not a numeric quantity, so one-hot encode it
X_train = pd.get_dummies(train_df[feature_cols], columns=["community_area"])
X_val = pd.get_dummies(val_df[feature_cols], columns=["community_area"])
X_test = pd.get_dummies(test_df[feature_cols], columns=["community_area"])

# Keep the dummy columns before scaling turns X_train into a plain array
train_columns = X_train.columns

# Make sure val/test have the same dummy columns as train (in case a community_area is missing)
X_val = X_val.reindex(columns=train_columns, fill_value=0)
X_test = X_test.reindex(columns=train_columns, fill_value=0)

y_train = train_df[TARGET_COL]
y_val = val_df[TARGET_COL]
y_test = test_df[TARGET_COL]

# SVR is distance-based, so all features need to be on a similar scale
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

print("Features:", feature_cols)
print("Target:", y_train.dtypes)

Features: ['month_sin', 'month_cos', 'weekday_sin', 'weekday_cos', 'hour_sin', 'hour_cos', 'tmpc', 'relh', 'sknt', 'vsby', 'p01m', 'skyc1_BKN', 'skyc1_CLR', 'skyc1_FEW', 'skyc1_OVC', 'skyc1_SCT', 'skyc1_VV ', 'is_holiday', 'community_area', 'food_drink', 'landmark', 'shop', 'train_station']
Target: uint32


In [45]:
# Create X and y for grid search (same encoding + scaling as the full training set)
X_train_grid = pd.get_dummies(train_df_grid[feature_cols], columns=["community_area"])
X_train_grid = X_train_grid.reindex(columns=train_columns, fill_value=0)
X_train_grid = scaler.transform(X_train_grid)

y_train_grid = train_df_grid[TARGET_COL]

In [46]:
model = SVR()

In [ ]:
param_grid_linear = {
    "C": [0.1, 1, 10],
    "epsilon": [0.1, 0.5, 1, 1.5],
    "kernel": ["linear"]
}

param_grid_rbf_sigmoid = {
    "C": [0.1, 1, 10],
    "epsilon": [0.1, 0.5, 1, 1.5],
    "kernel": ["rbf", "sigmoid"],
    "gamma": ["scale", "auto", 0.01, 0.1, 1]
}

param_grid_poly = {
    "C": [0.1, 1, 10],
    "epsilon": [0.01, 0.1, 0.5, 1],
    "kernel": ["poly"],
    "degree": [3, 4],
    "gamma": ["scale", "auto", 0.01, 0.1, 1]
}

grids = {}
for name, grid in [("linear", param_grid_linear), ("rbf_sigmoid", param_grid_rbf_sigmoid), ("poly", param_grid_poly)]:
    search = GridSearchCV(
        estimator=SVR(),
        param_grid=grid,
        cv=3,
        scoring="r2",
        n_jobs=-1,
        error_score="raise"
    )
    search.fit(X_train_grid, y_train_grid)
    grids[name] = search
    print(name, "best score:", search.best_score_, "best params:", search.best_params_)

best_name = max(grids, key=lambda name: grids[name].best_score_)
grid_search = grids[best_name]
print("Overall best:", best_name, grid_search.best_params_)

linear best score: 0.6166038771776193 best params: {'C': 1, 'epsilon': 1.5, 'kernel': 'linear'}
rbf_sigmoid best score: 0.5757606994989241 best params: {'C': 10, 'epsilon': 1.5, 'gamma': 'scale', 'kernel': 'rbf'}
poly best score: 0.8301373754704283 best params: {'C': 0.1, 'degree': 3, 'epsilon': 1, 'gamma': 0.1, 'kernel': 'poly'}
Overall best: poly {'C': 0.1, 'degree': 3, 'epsilon': 1, 'gamma': 0.1, 'kernel': 'poly'}


In [48]:
# Best parameters
print("Best parameters:", grid_search.best_params_)
print("Best CV score:", grid_search.best_score_)

Best parameters: {'C': 0.1, 'degree': 3, 'epsilon': 1, 'gamma': 0.1, 'kernel': 'poly'}
Best CV score: 0.8301373754704283


In [49]:
best_model = grid_search.best_estimator_

In [ ]:
# Train SVR 
best_model.fit(X_train, y_train)

In [ ]:
# Make prediction 
y_pred = best_model.predict(X_test)

In [ ]:
y_pred

array([ 2.04641932,  1.22728575,  0.74182025, ...,  1.73639758,
        0.54061983, -0.3537835 ], shape=(237252,))

In [ ]:
# Evaluation metrics

print("MAE:", mean_absolute_error(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2 Score:", r2_score(y_test, y_pred))

MAE: 3.8083307964510738
MSE: 259.4285489509462
RMSE: 16.10678580446596
R2 Score: 0.7850709894441041


In [ ]:
# save model
dump(best_model, "../models/model_" + SPATIAL_UNIT + "_svr.joblib")

['../models/model_community_svr.joblib']